# Lesson 25 — Robust Optimization

## Learning objectives

1. State the robust-counterpart paradigm.
2. Recognize box, ellipsoidal, and budget uncertainty sets.
3. Reformulate robust LPs as deterministic problems.
4. Distinguish static from adjustable RO.

## 1. Robust counterparts

For an LP $\min c^\top x$ s.t. $a_i^\top x \le b_i$ where $a_i \in \mathcal{U}_i$ (uncertainty set centred on the *nominal* $\hat a_i$), the **robust counterpart** is

$$\min c^\top x \;\;\text{s.t.}\;\; a_i^\top x \le b_i \;\; \forall a_i \in \mathcal{U}_i.$$

Equivalent: $\sup_{a \in \mathcal U_i} a^\top x \le b_i$, a *semi-infinite* constraint. Tractability hinges on reformulating this supremum {cite:p}`BenTal2009,BertsimasSim2004`.

> **Notation note.** This lesson uses $\hat a$ for the *nominal* (centre) value of an uncertain coefficient — distinct from $\overline a / \underline a$ which the rest of the course reserves for upper / lower *bounds*.

## 2. Common uncertainty sets

- **Box** $\{a : |a_i - \hat a_i| \le \delta_i\}$ → reformulates to LP. Conservative.
- **Ellipsoid** $\{\hat a + Pu : \|u\| \le 1\}$ → SOCP {cite:p}`BenTal1999`.
- **Budget** {cite:p}`BertsimasSim2004` $\{a : \sum_i |a_i - \hat a_i|/\delta_i \le \Gamma\}$ → LP. Adjustable conservatism via $\Gamma$.

## 3. Robust reformulations

**Box uncertainty:**

$$\sup_{|\Delta_i| \le \delta_i} (\hat a + \Delta)^\top x = \hat a^\top x + \sum_i \delta_i |x_i| \le b.$$

Linearize $|x_i|$ with $x = x^+ - x^-, x^\pm \ge 0$.

**Ellipsoidal:**

$$\hat a^\top x + \|P^\top x\|_2 \le b.$$

SOCP.

**Budget {cite:p}`BertsimasSim2004`:**

$$\hat a^\top x + \max_{S \subseteq [n], |S| \le \Gamma} \sum_{i \in S} \delta_i |x_i| \le b.$$

Reformulates to an LP with $O(n)$ extra variables.

## 4. Adjustable RO

**Static RO:** $x$ chosen before $\xi$ realized.
**Adjustable RO:** $y(\xi)$ chosen after $\xi$, can depend on it. Affine decision rules $y(\xi) = y_0 + Y\xi$ keep tractability {cite:p}`BenTalGoryashko2004`.

Multi-stage RO is harder but surfaces in inventory, energy, and scheduling problems.

In [1]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [2]:
# Robust LP: min c'x s.t. a'x >= b for all a in box around the nominal hat_a.
# Since x is free in sign here, we keep the |x| linearisation x = x+ - x-.
# (If you constrain x >= 0, |x_i| = x_i and you can drop x+/x-.)
import numpy as np, discopt as do

n = 5
np.random.seed(0)
hat_a = np.random.uniform(1, 5, size=n)            # nominal coefficients
delta = 0.2 * hat_a                                 # 20 % uncertainty
b = 30.0
c = np.random.uniform(1, 3, size=n)

m = do.Model("robust_lp")
xp = m.continuous("xp", shape=(n,), lb=0, ub=10)    # x = xp - xm, |x_i| = xp+xm
xm = m.continuous("xm", shape=(n,), lb=0, ub=10)
x = xp - xm
m.subject_to(hat_a @ x - delta @ (xp + xm) >= b)
m.minimize(c @ x)
r = m.solve()
print("status:", r.status, " robust obj:", round(r.objective, 4),
      " x:", (r.value(xp) - r.value(xm)).round(2))


status: optimal  robust obj: 18.0719  x: [ 0.   10.    0.   -0.23  0.  ]


## 5. RO vs SP

|  | RO | SP |
|--|----|----|
| Uncertainty | Worst case in $\mathcal U$ | Probability distribution |
| Solution | Min over $x$, max over $a$ | Min expected cost |
| Tractability | Often LP/SOCP | LP with many scenarios |
| Conservatism | Can be high; tunable via $\Gamma$ | Tunable via risk measure |

A mature robust optimizer reaches for both — RO when uncertainty is bounded but distributions are unknown, SP when distributions are reliable.

## References
{cite:p}`BenTal2009,BertsimasSim2004,BenTal1999,BenTalGoryashko2004,Bertsimas2010`.